# QuantumNet - Baselines over multiple seeds (CNN, ResNet50, MobileNetV2) + comparison with the hybrid
Before running: Add Input = APTOS 2019 Blindness Detection, Accelerator = GPU, Internet = On.
Run cells top to bottom. Start with MODELS = ['resnet50'] to check the time of one run, then add the others.

## Step 1: GPU check

In [1]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


## Step 2: Find dataset files automatically

In [2]:
import os, glob
csv_files = glob.glob('/kaggle/input/**/train.csv', recursive=True)
print(csv_files)
CSV_PATH = csv_files[0]
BASE = os.path.dirname(CSV_PATH)
IMG_DIR = os.path.join(BASE, 'train_images')
print(CSV_PATH); print(IMG_DIR, os.path.exists(IMG_DIR))

['/kaggle/input/competitions/aptos2019-blindness-detection/train.csv']
/kaggle/input/competitions/aptos2019-blindness-detection/train.csv
/kaggle/input/competitions/aptos2019-blindness-detection/train_images True


## Step 3: Binary labels + 70/15/15 split (seed 42)

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
SEED = 42
df = pd.read_csv(CSV_PATH)
print(df['diagnosis'].value_counts().sort_index())
df['label'] = (df['diagnosis'] > 0).astype(int)
print(df['label'].value_counts())
ext = os.listdir(IMG_DIR)[0].split('.')[-1]
df['path'] = IMG_DIR + '/' + df['id_code'] + '.' + ext
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)
print(len(train_df), len(val_df), len(test_df))   # expect 2563 549 550
os.makedirs('/kaggle/working', exist_ok=True)
for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    d.to_csv(f'/kaggle/working/{name}_split.csv', index=False)

diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64
label
1    1857
0    1805
Name: count, dtype: int64
2563 549 550


## Step 4: Dataset + augmentation

In [4]:
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
torch.manual_seed(SEED); np.random.seed(SEED)
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(20),
    transforms.RandomAffine(0, translate=(0.1, 0.1), shear=10, scale=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)])
eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)])

class DRDataset(Dataset):
    def __init__(self, frame, tf):
        self.paths = frame['path'].tolist(); self.labels = frame['label'].tolist(); self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.tf(img), self.labels[i]

BS = 32
train_loader = DataLoader(DRDataset(train_df, train_tf), batch_size=BS, shuffle=True, num_workers=2)
val_loader   = DataLoader(DRDataset(val_df, eval_tf), batch_size=BS, num_workers=2)
test_loader  = DataLoader(DRDataset(test_df, eval_tf), batch_size=BS, num_workers=2)
x, y = next(iter(train_loader)); print(x.shape, y[:8])

torch.Size([32, 3, 224, 224]) tensor([1, 0, 1, 1, 1, 1, 0, 0])


## Models and training function

In [5]:
import torch.nn as nn, torchvision.models as models
from scipy.stats import wilcoxon
import json, time
criterion = nn.CrossEntropyLoss()

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 28 * 28, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 2))
    def forward(self, x): return self.classifier(self.features(x))

def build(name):
    if name == 'resnet50':
        m = models.resnet50(weights='IMAGENET1K_V2')
        for p in m.parameters(): p.requires_grad = False
        m.fc = nn.Linear(m.fc.in_features, 2); return m, m.fc.parameters()
    if name == 'mobilenetv2':
        m = models.mobilenet_v2(weights='IMAGENET1K_V1')
        for p in m.parameters(): p.requires_grad = False
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.last_channel, 2)); return m, m.classifier.parameters()
    m = SimpleCNN(); return m, m.parameters()

def run_epoch(net, loader, train, opt=None):
    net.train() if train else net.eval()
    correct, n = 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = net(x); loss = criterion(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            correct += (out.argmax(1) == y).sum().item(); n += len(y)
    return correct / n

def train_once(name, seed, epochs=8, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    net, params = build(name); net = net.to(device)
    opt = torch.optim.Adam(params, lr=lr)
    best_va, best_state = 0, None
    for ep in range(epochs):
        run_epoch(net, train_loader, True, opt)
        va = run_epoch(net, val_loader, False)
        if va > best_va:
            best_va = va; best_state = {k: v.clone() for k, v in net.state_dict().items()}
    net.load_state_dict(best_state)
    return best_va, run_epoch(net, test_loader, False)

## Run: pick models and seeds (first test with one model and one seed to see the time)

In [6]:
MODELS = ['resnet50']            # later: ['resnet50', 'mobilenetv2', 'simple_cnn']
SEEDS = [5]                      # later: [0, 1, 2, 3, 4, 5]

try:
    results
except NameError:
    results = {}
for name in MODELS:
    results.setdefault(name, {})
    for s in SEEDS:
        if s in results[name]: continue          # skip runs already done
        t0 = time.time()
        va, te = train_once(name, s)
        results[name][s] = te
        print(f'{name} seed {s}: val {va:.4f} test {te:.4f} ({time.time()-t0:.0f}s)')
        json.dump({n: {str(k): v for k, v in d.items()} for n, d in results.items()},
                  open('/kaggle/working/baselines_multiseed.json', 'w'), indent=2)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 216MB/s]


resnet50 seed 5: val 0.9435 test 0.9309 (1835s)


## Summary + Wilcoxon against the hybrid and the classical control (run after 6 seeds are done)
Note: the pairing is by seed index; with 6 seeds the smallest possible p-value is 0.03125.

In [7]:
HYBRID  = [0.9545, 0.9436, 0.9491, 0.9491, 0.9545, 0.9473]   # from Part 2 (4 qubits, 6 layers)
CLASSIC = [0.9509, 0.9509, 0.9527, 0.9545, 0.9473, 0.9582]   # from Part 2 (matched classical head)
for name, d in results.items():
    a = np.array([d[s] for s in sorted(d)])
    print(f'{name}: {len(a)} seeds, mean {a.mean():.4f} +/- {a.std():.4f}')
    if len(a) == 6:
        print('   vs hybrid   :', wilcoxon(a, HYBRID))
        print('   vs classical:', wilcoxon(a, CLASSIC))

resnet50: 1 seeds, mean 0.9309 +/- 0.0000


## After this: Save Version > Save & Run All, then send me the printed outputs.